# Copyright

<PRE>
Jelen iPython notebook a Budapesti Műszaki és Gazdaságtudományi Egyetemen tartott "Deep Learning a gyakorlatban Python és LUA alapon" tantárgy segédanyagaként készült.
A tantárgy honlapja: http://smartlab.tmit.bme.hu/oktatas-deep-learning
Deep Learning kutatás: http://smartlab.tmit.bme.hu/deep-learning
Jelen notebook Nicolas P. Rougier munkája alapján készült, melyet BSD licensz véd: http://www.labri.fr/perso/nrougier/downloads/mlp.py

A notebook bármely részének újra felhasználása, publikálása csak a szerzők írásos beleegyezése esetén megegengedett.

2026 (c) Gyires-Tóth Bálint (toth.b kukac tmit pont bme pont hu), Zainkó Csaba
</PRE>

# 2. feladat – Transfer learning gyakorlás: bablevél-betegségek felismerése

Az 1. feladatban végigkövetted, hogyan tanítunk egy előtanított MobileNetV2 hálót virágok felismerésére. Most **te írod meg** a legfontosabb lépéseket egy egészen más adathalmazon.

**Az adathalmaz:** *beans*, azaz ugandai babföldeken, mobiltelefonnal készült levélfotók **3 osztályban**: egészséges levél, illetve két levélbetegség (szögletes levélfoltosság és babrozsda). Mezőgazdasági szakértő nélkül sokszor nehéz megmondani, melyik betegségről van szó. Lássuk, mennyire tanulja meg ezt egy háló alig ezer képből!

### Teendőd
A notebookban **5 feladat** van (🟡 **FELADAT**). Mindegyiknél csak néhány sort kell írnod a `# Ide írd a kódodat` megjegyzés helyére. A többi cella kész, csak futtatnod kell.

Minden feladat után egy **ellenőrző cella** következik: ha ✅-t ír ki, jó úton jársz; ha hibát dob, olvasd el az üzenetét.

- Kapcsold be a GPU-t: *Runtime → Change runtime type → T4 GPU*.
- Várható idő: kb. **15 perc**, ebből a tanítás kb. 3 perc.
- Segítségként nyugodtan nézd meg az 1. notebook megfelelő celláit!

## 0. Előkészítés (kész kód, csak futtasd)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "⚠️ NINCS – kapcsold be a T4 GPU-t!")

SEED = 42
IMG_SIZE = 160
BATCH_SIZE = 32
EPOCHS_FEATURE = 5
EPOCHS_FINETUNE = 5
keras.utils.set_random_seed(SEED)

In [ ]:
# Az adathalmazt a Hugging Face-ről töltjük le (kb. 170 MB, néhány másodperc)
from datasets import load_dataset

hf = load_dataset("AI-Lab-Makerere/beans")   # train / validation / test részre eleve fel van osztva


def to_tf(split):
    # PIL-képek -> IMG_SIZE×IMG_SIZE-es numpy tömb -> tf.data adathalmaz
    images = np.stack([np.array(img.convert("RGB").resize((IMG_SIZE, IMG_SIZE))) for img in split["image"]])
    labels = np.array(list(split["labels"]), dtype="int64")
    return tf.data.Dataset.from_tensor_slices((images, labels))


train_raw, val_raw, test_raw = to_tf(hf["train"]), to_tf(hf["validation"]), to_tf(hf["test"])

HU = {"angular_leaf_spot": "szögletes levélfoltosság", "bean_rust": "babrozsda",
      "healthy": "egészséges"}
CLASS_NAMES = [HU.get(n, n) for n in hf["train"].features["labels"].names]
NUM_CLASSES = len(CLASS_NAMES)

print(f"Osztályok száma: {NUM_CLASSES}")
print("Osztályok:", CLASS_NAMES)
print(f"Tanító: {int(train_raw.cardinality())}, validációs: {int(val_raw.cardinality())}, "
      f"teszt: {int(test_raw.cardinality())} kép")

plt.figure(figsize=(12, 4.5))
shown = set()
i = 0
for image, label in train_raw:
    label = int(label)
    if label in shown:
        continue
    shown.add(label)
    i += 1
    plt.subplot(1, NUM_CLASSES, i)
    plt.imshow(image.numpy())
    plt.title(CLASS_NAMES[label], fontsize=9)
    plt.axis("off")
    if len(shown) == NUM_CLASSES:
        break
plt.suptitle("Egy-egy példa minden osztályból (terepi fotók, 160×160-ra kicsinyítve)")
plt.tight_layout()
plt.show()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def resize(image, label):
    return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label


train_ds = train_raw.map(resize, num_parallel_calls=AUTOTUNE).cache().shuffle(1000, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds = val_raw.map(resize, num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_raw.map(resize, num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)


def plot_history(histories, title):
    """Egy vagy több egymást követő tanítás pontosságát rajzolja ki."""
    acc, val_acc = [], []
    for h in histories:
        acc += h.history["accuracy"]
        val_acc += h.history["val_accuracy"]
    epochs = range(1, len(acc) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, acc, "o-", label="tanító")
    plt.plot(epochs, val_acc, "o-", label="validációs")
    if len(histories) > 1:
        plt.axvline(len(histories[0].history["accuracy"]) + 0.5, color="gray", ls="--", label="fine-tuning kezdete")
    plt.xlabel("epoch")
    plt.ylabel("pontosság")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

print("✅ Adatok előkészítve")

## 🟡 1. FELADAT – Adatbővítés

Készíts egy `data_augmentation` nevű `keras.Sequential` modellt, amely
- véletlenszerűen **vízszintesen és függőlegesen is** tükröz,
- és véletlenszerűen elforgat (pl. 0.2 mértékben).

💡 *Gondold végig:* a virágoknál csak vízszintesen tükröztünk, mert egy fejjel lefelé álló virágot nem fényképezünk. A közelről fotózott leveleknél viszont nincs kitüntetett „fent” és „lent”: a levél bármilyen szögben állhat a képen, a betegség foltjai pedig ugyanúgy néznek ki. Ezért itt a függőleges tükrözés is megengedett.

*Segítség:* `layers.RandomFlip(...)`, `layers.RandomRotation(...)`. A tükrözés módja lehet `"horizontal"`, `"vertical"` vagy `"horizontal_and_vertical"`.

In [ ]:
# 1. FELADAT: hozd létre a data_augmentation modellt (kb. 4 sor)
# Ide írd a kódodat


In [ ]:
# Ellenőrzés (kész kód)
assert "data_augmentation" in globals(), "Nincs data_augmentation nevű változó – futtattad a feladat celláját?"
sample = next(iter(train_ds))[0][:1]
out = data_augmentation(sample, training=True)
assert tuple(out.shape) == (1, IMG_SIZE, IMG_SIZE, 3), f"Hibás kimeneti alak: {out.shape}"

plt.figure(figsize=(10, 2.5))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(np.clip(np.array(data_augmentation(sample, training=True)[0]) / 255, 0, 1))
    plt.axis("off")
plt.suptitle("Ugyanaz a kép 5 véletlen módosítással")
plt.show()
print("✅ 1. feladat rendben")

## 🟡 2. FELADAT – Az előtanított törzs betöltése

Töltsd be a `base_model` nevű változóba a **MobileNetV2** hálót
- `IMG_SIZE × IMG_SIZE × 3` méretű bemenettel,
- az **ImageNeten** előtanított súlyokkal,
- az eredeti osztályozó réteg **nélkül**,

majd **fagyaszd be** a súlyait.

*Segítség:* `keras.applications.MobileNetV2(input_shape=..., include_top=..., weights=...)`, illetve a `trainable` attribútum.

In [ ]:
# 2. FELADAT: töltsd be és fagyaszd be a MobileNetV2-t (kb. 2 utasítás)
# Ide írd a kódodat


In [ ]:
# Ellenőrzés (kész kód)
assert "base_model" in globals(), "Nincs base_model nevű változó!"
assert base_model.output.shape[-1] == 1280, "A kimenet nem 1280 csatornás. Biztosan include_top=False?"
assert not base_model.trainable, "A törzs nincs befagyasztva! (trainable)"
assert any("Conv1" == l.name for l in base_model.layers), "Ez nem MobileNetV2?"
print("Törzs kimenete:", base_model.output.shape)
print("✅ 2. feladat rendben")

## 🟡 3. FELADAT – A teljes modell összeállítása

A modell eleje már kész. Egészítsd ki az alábbi lépésekkel:
1. a `base_model` meghívása, **inference módban** (`training=False`),
2. `GlobalAveragePooling2D`,
3. `Dropout(0.2)`,
4. kimeneti `Dense` réteg: **hány neuron** kell, és milyen aktiváció?

⚠️ Itt **3 osztály** van, nem 5, mint a virágoknál! Ne írd be fixen a számot, használd a `NUM_CLASSES` változót.

A kimenetet tedd az `outputs` változóba.

In [ ]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)   # a MobileNetV2 [-1, 1] tartományú bemenetet vár

# 3. FELADAT: egészítsd ki a modellt (kb. 4 sor), a végeredmény az `outputs` változó legyen
# Ide írd a kódodat


model = keras.Model(inputs, outputs, name="szovettani_mobilenetv2")

In [ ]:
# Ellenőrzés (kész kód)
assert model.outputs[0].shape[-1] == NUM_CLASSES, \
    f"A kimeneten {model.outputs[0].shape[-1]} neuron van, de {NUM_CLASSES} osztályunk van!"
assert getattr(model.layers[-1].activation, "__name__", "") == "softmax", \
    "Az utolsó réteg aktivációja nem softmax!"
assert len(model.trainable_weights) == 2, \
    "Túl sok tanítható súly: a törzs nincs befagyasztva, vagy több Dense réteget adtál hozzá?"
n_train = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
n_all = n_train + sum(int(np.prod(w.shape)) for w in model.non_trainable_weights)
print(f"Tanítható paraméterek: {n_train:,} / {n_all:,} ({100 * n_train / n_all:.2f}%)")
print("✅ 3. feladat rendben")

## 🟡 4. FELADAT – Feature extraction: fordítás és tanítás

1. Fordítsd le a modellt (`compile`): **Adam** optimalizáló `1e-3` tanulási rátával, `"sparse_categorical_crossentropy"` veszteségfüggvény, `"accuracy"` metrika.
2. Tanítsd `EPOCHS_FEATURE` epochig a `train_ds` adatokon, `val_ds` validációval. A `fit` visszatérési értékét tedd a `history_fe` változóba.

In [ ]:
# 4. FELADAT: compile + fit (kb. 2 utasítás)
# Ide írd a kódodat


In [ ]:
# Ellenőrzés és kiértékelés (kész kód)
assert "history_fe" in globals(), "Nincs history_fe változó – a fit eredményét abba tedd!"
plot_history([history_fe], "Feature extraction")
_, fe_acc = model.evaluate(test_ds, verbose=0)
print(f"Teszt pontosság feature extraction után: {fe_acc:.1%}  (véletlen tippelés: {1 / NUM_CLASSES:.1%})")
if fe_acc < 0.5:
    print("⚠️ Gyanúsan alacsony – ellenőrizd a tanulási rátát és a 3. feladatot!")
else:
    print("✅ 4. feladat rendben")

## 🟡 5. FELADAT – Fine-tuning

1. Engedd fel a törzset (`base_model.trainable = True`), majd az **első 100 rétegét** fagyaszd vissza.
2. Fordítsd újra a modellt, ezúttal **100-szor kisebb** tanulási rátával (`1e-5`).
3. Folytasd a tanítást további `EPOCHS_FINETUNE` epochig. Ügyelj rá, hogy az epochszámozás onnan folytatódjon, ahol abbahagytad (`initial_epoch`). Az eredményt a `history_ft` változóba tedd.

*Segítség:* `base_model.layers[:100]` az első 100 réteg listája.

In [ ]:
# 5. FELADAT: fine-tuning (kb. 5-6 sor)
# Ide írd a kódodat


In [ ]:
# Ellenőrzés és kiértékelés (kész kód)
assert "history_ft" in globals(), "Nincs history_ft változó!"
n_unfrozen = sum(l.trainable for l in base_model.layers)
assert 0 < n_unfrozen < len(base_model.layers), \
    f"{n_unfrozen} réteg tanítható a {len(base_model.layers)}-ből – csak a felső rétegeket kellene felengedni!"
lr = float(keras.ops.convert_to_numpy(model.optimizer.learning_rate))
if lr > 1e-4:
    print(f"⚠️ A tanulási ráta {lr:g} – fine-tuninghoz ez túl nagy lehet!")

plot_history([history_fe, history_ft], "Feature extraction + fine-tuning")
_, ft_acc = model.evaluate(test_ds, verbose=0)
print(f"Teszt pontosság:  feature extraction után {fe_acc:.1%}  →  fine-tuning után {ft_acc:.1%}")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = model.predict(test_ds, verbose=0).argmax(axis=1)
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASS_NAMES).plot(
    ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
ax.set_xlabel("Becsült osztály")
ax.set_ylabel("Valós osztály")
plt.title("Konfúziós mátrix (teszthalmaz)")
plt.show()
print("✅ 5. feladat rendben – kész vagy a kódolással!")

## Rövid kérdések

Válaszolj 1–3 mondatban a „Válasz” cellákban!

**1.** Mit kellett megváltoztatnod a modell felépítésében az 1. notebookhoz képest, és miért? Mi történt volna, ha a kimeneti rétegben 5 neuron marad?

**Válasz:** *ide írd*

**2.** Miért volt értelme itt függőleges tükrözést is használni, a virágoknál viszont nem?

**Válasz:** *ide írd*

**3.** Hogyan alakult a teszt pontosság feature extraction és fine-tuning után? A konfúziós mátrix alapján melyik két osztályt keveri leginkább a modell, és miért segíthet ezen a fine-tuning?

**Válasz:** *ide írd*